# Training a Network with FLBO for Convolutions

This notebook demonstrates training a neural network using the FLBO (Filtering on Local Bounded Operators) method for convolutions. We will be utilizing the **TOSCA** dataset for this task.

⚠️ **Important:** If you are running this on Google Colab, please make sure to select a runtime with a GPU (Go to *Runtime* > *Change runtime type* > *Hardware accelerator* > *GPU*).

*Note: A pre-trained model is already available at your disposal if you prefer to skip the training phase and go straight to evaluation.*

### Environment Setup
First, we need to download the TOSCA dataset benchmarks and extract them. We will also clone the FLBO repository and install the required dependencies, such as `torch_geometric` and `libigl`.

In [ ]:
!wget -q https://vovakim.com/projects/CorrsBlended/SurfCorr2.0.benchmarks.bins.zip
!unzip -q SurfCorr2.0.benchmarks.bins.zip
!mkdir -p /content/TOSCA/raw/
!cp -r CorrsBenchmark/Data/TOSCA/Meshes/* /content/TOSCA/raw/

!git clone -q https://github.com/tpitois/FLBO
%cd FLBO
!pip install -q torch_geometric libigl

In [ ]:
import os, json
import torch
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from time import time

from src.datasets.TOSCA import TOSCA
from src.datasets.transforms import FLBOTransform
from src.models.ascnn import ACSCNN
from src.utils.eval import evaluate_predictions
from src.utils.viz import plot_pck_curve, plot_correspondence, plot_learning_curves
import matplotlib.pyplot as plt

Here, we load the dataset apply the FLBO transformation which computes the FLBO and we also decimate meshes so the training be faster.

In [ ]:
dataset = TOSCA(
    root='../data/TOSCA/',
    categories='cat',
    pre_transform=FLBOTransform(n_angles=8, alpha=10.0, tau=0.5)
)

train_dataset = dataset[:9]
test_dataset = dataset[9:]

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

n_desc = dataset[0].x.size(1)
n_class = dataset[0].num_nodes

model = ACSCNN(n_desc, n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
criterion = torch.nn.CrossEntropyLoss()

num_epochs = 200

scaler = torch.amp.GradScaler('cuda')

In [ ]:
save_dir = Path("../results")
os.makedirs(save_dir, exist_ok=True)

history = {
    'train_loss': [],
    'train_acc': [],
}

all_epoch_preds = []
all_epoch_labels = []

### Training Loop
We utilize PyTorch's Automatic Mixed Precision (`torch.amp.autocast`) to optimize memory usage and calculation speed. Once training is complete, the model's state and the training history are saved.

In [ ]:
pbar = tqdm(range(num_epochs), leave=True)

for epoch in pbar:
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_nodes = 0

    epoch_preds = []
    epoch_labels = []

    for i, data in enumerate(train_loader):
        x = data.x.to(device)
        L = data.L.to(device)
        y = data.y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(x, L)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item()
        running_corrects += torch.sum(preds == y).item()
        total_nodes += y.size(0)


    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_corrects / total_nodes

    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)

    pbar.set_postfix({
        'loss': f'{epoch_loss:.4f}',
        'acc': f'{epoch_acc:.4f}'
    })

torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': epoch_loss,
}, os.path.join(save_dir, f'model{time()}.pth'))

with open(save_dir / Path('training_history{time()}.json'), 'w') as f:
    json.dump(history, f, indent=4)

### Loading the Trained Model
After training, we load the saved model checkpoint and training history from our results directory so we can evaluate its performance.

In [ ]:
checkpoint = torch.load(save_dir / Path('model.pth'), map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

with open(save_dir / Path('training_history.json'), 'r') as f:
        history = json.load(f)

### Visualizing Training Progress
Plot the learning curves to see how the loss and accuracy evolved over the training epochs.

In [ ]:
plot_learning_curves(history)

### Evaluation on the Test Set
For each mesh, we compute the predictions and calculate the errors against the ground truth labels to evaluate our model's accuracy.

In [ ]:
model.eval()

errors_list = []
mesh_list = []
preds_list = []

for i, data in enumerate(test_loader):
    x = data.x.to(device)
    L = data.L.to(device)
    y = data.y.cpu().numpy()

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            outputs = model(x, L)
            _, preds = torch.max(outputs, 1)

    preds = preds.cpu().numpy()

    V = data.pos.numpy()
    F = data.face.t().numpy()

    errors_list.append(evaluate_predictions(preds, y, V, F, num_samples=1000))
    mesh_list.append((V, F))
    preds_list.append(preds)

### PCK Curve
We plot the Percentage of Correct Keypoints (PCK) curve.

In [ ]:
plot_pck_curve(errors_list, max_threshold=0.15)
plt.show()

### Correspondence Visualization
We'll plot the 3D meshes side-by-side to visualize the predicted correspondences for the two examples of test set.

In [ ]:
plotter = plot_correspondence(*(mesh_list[0]), preds_list[0])
plotter.show()

In [ ]:
plotter = plot_correspondence(*(mesh_list[1]), preds_list[1])
plotter.show()